<!--BOOK_INFORMATION-->
<img align="left" style="padding-right:10px;" src="figures/PDSH-cover-small.png">
*This notebook contains an excerpt from the [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) by Jake VanderPlas; the content is available [on GitHub](https://github.com/jakevdp/PythonDataScienceHandbook).*

*The text is released under the [CC-BY-NC-ND license](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), and code is released under the [MIT license](https://opensource.org/licenses/MIT). If you find this content useful, please consider supporting the work by [buying the book](http://shop.oreilly.com/product/0636920034919.do)!*

<!--NAVIGATION-->
< [Hierarchical Indexing](03.05-Hierarchical-Indexing.ipynb) | [Contents](Index.ipynb) | [Combining Datasets: Merge and Join](03.07-Merge-and-Join.ipynb) >

# 3.7 Combining Datasets: Concat and Append 
# 合并数据集

Some of the most interesting studies of data come from combining different data sources.
These operations can involve anything from very straightforward concatenation of two different datasets, to more complicated database-style joins and merges that correctly handle any overlaps between the datasets.
``Series`` and ``DataFrame``s are built with this type of operation in mind, and Pandas includes functions and methods that make this sort of data wrangling fast and straightforward.

Here we'll take a look at simple concatenation of ``Series`` and ``DataFrame``s with the ``pd.concat`` function; later we'll dive into more sophisticated in-memory merges and joins implemented in Pandas.

We begin with the standard imports:

In [1]:
import pandas as pd
import numpy as np

For convenience, we'll define this function which creates a ``DataFrame`` of a particular form that will be useful below:

In [4]:
# 🐍 定义函数make_df, 快速生成一个DateFrame数组

def make_df(cols, ind):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in ind]
            for c in cols}
    return pd.DataFrame(data, ind)

# example DataFrame
make_df('ABC', range(3))

,A,B,C
0,A0,B0,C0
1,A1,B1,C1
2,A2,B2,C2


In addition, we'll create a quick class that allows us to display multiple ``DataFrame``s side by side. The code makes use of the special ``_repr_html_`` method, which IPython uses to implement its rich object display:

In [5]:
class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)
    

The use of this will become clearer as we continue our discussion in the following section.

## 3.7.1 Recall: Concatenation of NumPy Arrays 
## 知识回顾：Numpy数组的合并

Concatenation of ``Series`` and ``DataFrame`` objects is very similar to concatenation of Numpy arrays, which can be done via the ``np.concatenate`` function as discussed in [The Basics of NumPy Arrays](02.02-The-Basics-Of-NumPy-Arrays.ipynb).
Recall that with it, you can combine the contents of two or more arrays into a single array:

In [6]:
# 🐍 np.concatenate()函数

x = [1, 2, 3]
y = [4, 5, 6]
z = [7, 8, 9]
np.concatenate([x, y, z])

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

The first argument is a list or tuple of arrays to concatenate.
Additionally, it takes an ``axis`` keyword that allows you to specify the axis along which the result will be concatenated:

In [9]:
# 🐍 np.concatenate()函数
# 🐍 第一个参数是数据列表或元组，第二个参数axis是合并的坐标轴方向，
# 🐍 axis默认为行 (axis=0), axis=1 代表按列合并

x = [[1, 2],
     [3, 4]]
np.concatenate([x, x], axis=1)

array([[1, 2, 1, 2],
       [3, 4, 3, 4]])

## 3.7.2 Simple Concatenation with ``pd.concat``
## 通过 pd.concat 实现简易合并

Pandas has a function, ``pd.concat()``, which has a similar syntax to ``np.concatenate`` but contains a number of options that we'll discuss momentarily:

```python
# Signature in Pandas v0.18
pd.concat(objs, axis=0, join='outer', join_axes=None, ignore_index=False,
          keys=None, levels=None, names=None, verify_integrity=False,
          copy=True)
```

``pd.concat()`` can be used for a simple concatenation of ``Series`` or ``DataFrame`` objects, just as ``np.concatenate()`` can be used for simple concatenations of arrays:

In [13]:
# 🐍 np.concatenate()函数合并一维Series或DataFrame对象

ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([ser1, ser2], axis=1)

,0,1
1,A,NaN
2,B,NaN
3,C,NaN
4,NaN,D
5,NaN,E
6,NaN,F


It also works to concatenate higher-dimensional objects, such as ``DataFrame``s:

In [15]:
# 🐍 np.concatenate()函数合并高维数据

df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
display('df1', 'df2', 'pd.concat([df1, df2], axis=1).fillna(0)')

df1
    A   B
1  A1  B1
2  A2  B2

df2
    A   B
3  A3  B3
4  A4  B4

pd.concat([df1, df2], axis=1).fillna(0)
    A   B   A   B
1  A1  B1   0   0
2  A2  B2   0   0
3   0   0  A3  B3
4   0   0  A4  B4

By default, the concatenation takes place row-wise within the ``DataFrame`` (i.e., ``axis=0``).
Like ``np.concatenate``, ``pd.concat`` allows specification of an axis along which concatenation will take place.
Consider the following example:

🐍 默认情况下，DataFrame的合并是逐行进行的 (默认axis=0)

In [17]:
# 🐍 axis='col' 已失效，axis=1按列合并

df3 = make_df('AB', [0, 1])
df4 = make_df('AB', [0, 1])
display('df3', 'df4', "pd.concat([df3, df4], axis=0)")

,A,B
0,A0,B0
1,A1,B1
,A,B
0,A0,B0
1,A1,B1
,A,B
0,A0,B0
1,A1,B1
0,A0,B0
1,A1,B1


We could have equivalently specified ``axis=1``; here we've used the more intuitive ``axis='col'``. 

### 1. Duplicate indices / 索引重复

One important difference between ``np.concatenate`` and ``pd.concat`` is that Pandas concatenation *preserves indices*, even if the result will have duplicate indices!
Consider this simple example:

In [18]:
# 🐍 np.concatenate 与 pd.concat 最主要差异在于，Pandas在合并时会保留索引

x = make_df('AB', [0, 1])
y = make_df('AB', [2, 3])
y.index = x.index  # make duplicate indices!
display('x', 'y', 'pd.concat([x, y])')

,A,B
0,A0,B0
1,A1,B1
,A,B
0,A2,B2
1,A3,B3
,A,B
0,A0,B0
1,A1,B1
0,A2,B2
1,A3,B3


Notice the repeated indices in the result.
While this is valid within ``DataFrame``s, the outcome is often undesirable.
``pd.concat()`` gives us a few ways to handle it.

#### (1) Catching the repeats as an error / 捕捉索引重复的错误

If you'd like to simply verify that the indices in the result of ``pd.concat()`` do not overlap, you can specify the ``verify_integrity`` flag.
With this set to True, the concatenation will raise an exception if there are duplicate indices.
Here is an example, where for clarity we'll catch and print the error message:

In [19]:
# 🐍 参数 verify_integrity 设置，重复索引可触发异常

try:
    pd.concat([x, y], verify_integrity=True)
except ValueError as e:
    print("ValueError:", e)

ValueError: Indexes have overlapping values: Index([0, 1], dtype='int64')


#### (2) Ignoring the index / 忽略索引

Sometimes the index itself does not matter, and you would prefer it to simply be ignored.
This option can be specified using the ``ignore_index`` flag.
With this set to true, the concatenation will create a new integer index for the resulting ``Series``:

In [20]:
# 🐍 参数 ingnor_index 设置，重复索引可忽略

display('x', 'y', 'pd.concat([x, y], ignore_index=True)')

,A,B
0,A0,B0
1,A1,B1
,A,B
0,A2,B2
1,A3,B3
,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3


#### (3) Adding MultiIndex keys / 增加多级索引

Another option is to use the ``keys`` option to specify a label for the data sources; the result will be a hierarchically indexed series containing the data:

In [21]:
# 🐍 参数 keys 设置，多级索引标签

display('x', 'y', "pd.concat([x, y], keys=['a', 'b'])")

x
    A   B
0  A0  B0
1  A1  B1

y
    A   B
0  A2  B2
1  A3  B3

pd.concat([x, y], keys=['a', 'b'])
      A   B
a 0  A0  B0
  1  A1  B1
b 0  A2  B2
  1  A3  B3

The result is a multiply indexed ``DataFrame``, and we can use the tools discussed in [Hierarchical Indexing](03.05-Hierarchical-Indexing.ipynb) to transform this data into the representation we're interested in.

### 2. Concatenation with joins / 类似join的合并

In the simple examples we just looked at, we were mainly concatenating ``DataFrame``s with shared column names.
In practice, data from different sources might have different sets of column names, and ``pd.concat`` offers several options in this case.
Consider the concatenation of the following two ``DataFrame``s, which have some (but not all!) columns in common:

🐍 列名部分相同的情况

In [24]:
# 🐍 默认情况下，缺失的数据用NaN表示

df5 = make_df('ABC', [1, 2])
df6 = make_df('BCA', [3, 4])
display('df5', 'df6', 'pd.concat([df5, df6])')

,A,B,C
1,A1,B1,C1
2,A2,B2,C2
,B,C,A
3,B3,C3,A3
4,B4,C4,A4
,A,B,C
1,A1,B1,C1
2,A2,B2,C2
3,A3,B3,C3
4,A4,B4,C4


By default, the entries for which no data is available are filled with NA values.
To change this, we can specify one of several options for the ``join`` and ``join_axes`` parameters of the concatenate function.
By default, the join is a union of the input columns (``join='outer'``), but we can change this to an intersection of the columns using ``join='inner'``:

In [25]:
# 🐍 参数 join 设置：默认对 列 进行 并集合并，join='outer'

display('df5', 'df6',
        "pd.concat([df5, df6], join='outer')")

,A,B,C
1,A1,B1,C1
2,A2,B2,C2
,B,C,A
3,B3,C3,A3
4,B4,C4,A4
,A,B,C
1,A1,B1,C1
2,A2,B2,C2
3,A3,B3,C3
4,A4,B4,C4


In [26]:
# 🐍 join='inner' 实现对 列 进行 交集合并

display('df5', 'df6',
        "pd.concat([df5, df6], join='inner')")

,A,B,C
1,A1,B1,C1
2,A2,B2,C2
,B,C,A
3,B3,C3,A3
4,B4,C4,A4
,A,B,C
1,A1,B1,C1
2,A2,B2,C2
3,A3,B3,C3
4,A4,B4,C4


Another option is to directly specify the index of the remaininig colums using the ``join_axes`` argument, which takes a list of index objects.
Here we'll specify that the returned columns should be the same as those of the first input:

In [27]:
# 🐍 参数 join_axes 设置, 直接确定合并使用的列名, 索引对象构成的列表(列表的列表)
# 🐍 参数 join_axes 在较新版本已不适用，使用join+axis代替

display('df5', 'df6',
        "pd.concat([df5, df6], join_axes=[df5.columns])")

TypeError: concat() got an unexpected keyword argument 'join_axes'

TypeError: concat() got an unexpected keyword argument 'join_axes'

The combination of options of the ``pd.concat`` function allows a wide range of possible behaviors when joining two datasets; keep these in mind as you use these tools for your own data.

### 3. The ``append()`` method / append()方法

Because direct array concatenation is so common, ``Series`` and ``DataFrame`` objects have an ``append`` method that can accomplish the same thing in fewer keystrokes.
For example, rather than calling ``pd.concat([df1, df2])``, you can simply call ``df1.append(df2)``:

In [28]:
# 🐍 append()方法不存在 --> pd.concat()方法
# display('df1', 'df2', 'df1.append(df2)')

df1.append(df2)

AttributeError: 'DataFrame' object has no attribute 'append'

Keep in mind that unlike the ``append()`` and ``extend()`` methods of Python lists, the ``append()`` method in Pandas does not modify the original object–instead it creates a new object with the combined data.

🐍 Pandas的append()不直接更新原有对象的值，每次合并后创建一个新对象 --> 已弃用

It also is not a very efficient method, because it involves creation of a new index *and* data buffer.
Thus, if you plan to do multiple ``append`` operations, it is generally better to build a list of ``DataFrame``s and pass them all at once to the ``concat()`` function.

In the next section, we'll look at another more powerful approach to combining data from multiple sources, the database-style merges/joins implemented in ``pd.merge``.
For more information on ``concat()``, ``append()``, and related functionality, see the ["Merge, Join, and Concatenate" section](http://pandas.pydata.org/pandas-docs/stable/merging.html) of the Pandas documentation.

<!--NAVIGATION-->
< [Hierarchical Indexing](03.05-Hierarchical-Indexing.ipynb) | [Contents](Index.ipynb) | [Combining Datasets: Merge and Join](03.07-Merge-and-Join.ipynb) >